In [0]:
WITH monthly_sales AS (
  SELECT
    DATE_TRUNC('MONTH', s.BillingDate) AS month,
    s.SalesOrganization,
    i.MaterialGroup,
    SUM(i.NetAmount) AS total_sales
  FROM
    `jeffdemo`.`dlife`.`v_sales_document_item_02` i
      JOIN (
        SELECT DISTINCT
          BillingDate,
          SalesOrganization,
          OriginSDDocument,
          OriginSDDocumentItem
        FROM
          `jeffdemo`.`dlife`.`v_billing_document_item_01`
        WHERE
          SalesOrganization IN ('PH01', 'PH02', 'MD01')
      ) s
        ON i.OriginSDDocument = s.OriginSDDocument
        AND i.OriginSDDocumentItem = s.OriginSDDocumentItem
  WHERE
    i.MaterialGroup IN ('M001', 'N001', 'L001')
    AND s.BillingDate IS NOT NULL
    AND i.NetAmount IS NOT NULL
    AND s.BillingDate >= '2022-01-01'
    AND s.BillingDate <= '2026-08-31'
  GROUP BY
    1,
    2,
    3
),
seasonality AS (
  SELECT
    month,
    SalesOrganization,
    SUM(total_sales) AS org_monthly_sales
  FROM
    monthly_sales
  GROUP BY
    1,
    2
),
growth AS (
  SELECT
    month,
    MaterialGroup,
    SUM(total_sales) AS group_monthly_sales,
    LAG(SUM(total_sales)) OVER (PARTITION BY MaterialGroup ORDER BY month) AS prev_month_sales,
    TRY_DIVIDE(
      SUM(total_sales) - LAG(SUM(total_sales)) OVER (PARTITION BY MaterialGroup ORDER BY month),
      LAG(SUM(total_sales)) OVER (PARTITION BY MaterialGroup ORDER BY month)
    )
    * 100 AS growth_rate_pct
  FROM
    monthly_sales
  GROUP BY
    1,
    2
),
avg_peak_base AS (
  SELECT
    SalesOrganization,
    MaterialGroup,
    AVG(total_sales) AS avg_monthly_sales,
    MAX(total_sales) AS peak_month_sales
  FROM
    monthly_sales
  GROUP BY
    1,
    2
),
peak_months AS (
  SELECT
    SalesOrganization,
    MaterialGroup,
    month,
    total_sales,
    ROW_NUMBER() OVER (
        PARTITION BY SalesOrganization, MaterialGroup
        ORDER BY total_sales DESC
      ) AS rn
  FROM
    monthly_sales
),
avg_peak AS (
  SELECT
    a.SalesOrganization,
    a.MaterialGroup,
    a.avg_monthly_sales,
    a.peak_month_sales,
    p.month AS peak_month
  FROM
    avg_peak_base a
      LEFT JOIN peak_months p
        ON a.SalesOrganization = p.SalesOrganization
        AND a.MaterialGroup = p.MaterialGroup
        AND p.rn = 1
)
SELECT
  m.month,
  m.SalesOrganization,
  m.MaterialGroup,
  m.total_sales,
  s.org_monthly_sales,
  g.group_monthly_sales,
  g.growth_rate_pct,
  a.avg_monthly_sales,
  a.peak_month_sales,
  a.peak_month
FROM
  monthly_sales m
    LEFT JOIN seasonality s
      ON m.month = s.month
      AND m.SalesOrganization = s.SalesOrganization
    LEFT JOIN growth g
      ON m.month = g.month
      AND m.MaterialGroup = g.MaterialGroup
    LEFT JOIN avg_peak a
      ON m.SalesOrganization = a.SalesOrganization
      AND m.MaterialGroup = a.MaterialGroup
ORDER BY
  m.month,
  m.SalesOrganization,
  m.MaterialGroup